# BA-05: Dashboard Ejecutivo de Supply Chain



## 📋 Contexto del Caso de Negocio

**Empresa:** "GlobalRetail Corp" - Empresa retail multinacional con operaciones en 15 países y 200 centros de distribución.

**Situación actual:**
- KPIs dispersos: Más de 30 indicadores en 8 sistemas diferentes
- **Problema:** Ejecutivos tardan 3+ horas en preparar reportes para reuniones semanales de S&OP
- Factores relevantes:
  - Información desactualizada (reportes manuales con 48h de rezago)
  - Falta de visión integrada causa decisiones reactivas en lugar de proactivas
  - Equipos no alineados en prioridades (ventas vs. operaciones vs. finanzas)

**Impacto financiero:**
- Pérdida de $2.5M mensuales por decisiones tardías en reabastecimiento
- 15 horas ejecutivas/semana gastadas en compilación manual de reportes
- Costo de oportunidad: falta de identificación temprana de problemas operacionales

**Objetivo:** Implementar un dashboard ejecutivo integrado para:
1. Consolidar 5 KPIs críticos en vista única (OTIF, Fill Rate, Inventory Turns, Costo Logístico, Perfect Order)
2. Reducir tiempo de preparación de reportes de 3h a 15 minutos
3. Proveer visibilidad en tiempo real del desempeño operacional
4. Facilitar toma de decisiones data-driven en reuniones de S&OP

### 💼 ¿Por qué es IMPORTANTE?
- **Velocidad en la toma de decisiones:** Líderes necesitan respuestas rápidas sin "navegar" múltiples sistemas
- **Alineación organizacional:** Un solo dashboard crea narrativa común entre departamentos
- **Identificación temprana de problemas:** Alertas visuales permiten acciones correctivas antes de impactos mayores
- **Comunicación efectiva con stakeholders:** Formato ejecutivo para board meetings y presentaciones

### 🎁 ¿PARA QUÉ sirve?
- **S&OP meetings semanales:** Revisión rápida de performance operacional
- **Board reporting mensual:** Comunicación de resultados a dirección
- **Evaluación de estrategia:** Validar si iniciativas están dando resultados esperados
- **Drill-down interactivo:** Navegación desde vista ejecutiva a detalles operacionales cuando se detectan anomalías

### 🔧 ¿CÓMO se implementa?
- **Datos requeridos:** Orders, Inventory, Transport events, Locations, Sales
- **Cálculo principal:** `5 KPIs core = OTIF + Fill Rate + Inventory Turns + Logistics Cost % + Perfect Order`
- **Métrica resultado:** Dashboard consolidado con indicadores de semáforo y tendencias
- **Técnica aplicada:** Business Intelligence con visualizaciones interactivas Plotly, diseño de métricas balanceadas (operacional + financiero + cliente)

---

## 🎯 Objetivos de Aprendizaje

- Calcular y visualizar los 5 KPIs críticos de supply chain ejecutivo
- Diseñar dashboards interactivos con Plotly para audiencias ejecutivas
- Interpretar métricas operacionales y su impacto financiero
- Implementar sistema de alertas visuales con código de colores
- Integrar múltiples fuentes de datos en vista consolidada

## 📦 Instalación de Librerías Necesarias

**Antes de ejecutar este notebook, asegúrate de tener instaladas todas las dependencias.**

### Opción 1: Instalación dentro del notebook
Ejecuta la siguiente celda para instalar las librerías necesarias:

```python
%pip install pandas numpy plotly
```

### Opción 2: Instalación desde terminal
Si prefieres instalar desde la terminal, ejecuta:

```bash
# PowerShell o CMD
pip install pandas numpy plotly

# O si usas el proyecto completo con pyproject.toml
pip install -e .[core,notebooks]
```

### Librerías requeridas:
- `pandas`: Manipulación y análisis de datos
- `numpy`: Cálculos numéricos y simulaciones
- `plotly`: Visualización interactiva de dashboards ejecutivos

---

### 📝 Información del Notebook

| Campo | Valor |
| :--- | :--- |
| **🆔 ID** | `BA-05` |
| **📛 Título** | `Dashboard Ejecutivo de Supply Chain` |
| **🔹 Especialidad** | `Business Analytics / BI` |
| **⚙️ Proceso** | `Deliver / Plan` |
| **🧠 Nivel** | `Intermediate` |
| **⏱️ Duración** | `45 min` |
| **🏷️ Etiquetas** | `dashboard`, `kpis`, `executive-reporting`, `plotly`, `bi` |

---

## ⚙️ Configuración Inicial

## 🎯 Contexto del Notebook

### ¿Qué?
Dashboard ejecutivo que consolida los 5 KPIs más críticos de supply chain en visualizaciones interactivas tipo semáforo con análisis de tendencias y benchmarks.

### ¿Por qué?
Ejecutivos pierden 3+ horas semanales compilando KPIs de múltiples sistemas. Decisiones se retrasan por falta de visión integrada. Equipos desalineados por métricas inconsistentes entre departamentos.

### ¿Para qué?
- Reducir tiempo de preparación de reportes S&OP de 3h a 15 minutos
- Identificar problemas operacionales en tiempo real (alertas visuales)
- Alinear organización con métricas comunes en board meetings
- Facilitar drill-down desde vista ejecutiva a análisis detallado

### ¿Cuándo?
- Actualización diaria automática (8:00 AM)
- Revisión semanal en reuniones S&OP (lunes 9:00 AM)
- Presentación mensual a board (primer viernes del mes)
- Refresh en vivo durante reuniones de crisis

### ¿Cómo?
1. Cargar datos de órdenes, inventario, transporte y ventas
2. Calcular 5 KPIs core: OTIF, Fill Rate, Inventory Turns, Logistics Cost %, Perfect Order
3. Crear visualizaciones ejecutivas con código de colores (verde/amarillo/rojo)
4. Integrar en dashboard consolidado con indicadores y sparklines
5. Generar análisis Pareto de problemas operacionales

In [13]:
# ⚙️ Configuración de rutas
import sys
from pathlib import Path

def resolve_repo_root():
    """Detecta raíz del repositorio buscando carpetas data/ y notebooks/"""
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / 'data').exists() and (parent / 'notebooks').exists():
            return parent
    return current

root = resolve_repo_root()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f"✅ Rutas configuradas: {root}")

✅ Rutas configuradas: f:\GitHub\supply-chain-data-notebooks


In [14]:
# 📚 Importar librerías
import pandas as pd
import numpy as np
import warnings
from datetime import datetime, timedelta

# Visualización
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Configuración
np.random.seed(42)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
warnings.filterwarnings('ignore')

print(f"✅ Librerías cargadas")
print(f"   pandas: {pd.__version__}")
print(f"   numpy: {np.__version__}")

✅ Librerías cargadas
   pandas: 2.3.3
   numpy: 2.3.3


---

## 📥 Paso 1: Cargar y Preparar Datos

**Técnica:** Ingesta de datos desde múltiples fuentes (órdenes, inventario, transporte)

**Parámetros clave:**
- `parse_dates`: Convertir columnas de fecha para análisis temporal
- Datasets requeridos: orders, inventory, transport_events, locations

**Validaciones:**
- Verificar completitud de datos (sin registros nulos críticos)
- Confirmar rango temporal coherente

**⚠️ Nota sobre simulación de datos:**
Los datasets base (orders, inventory, products) son reales del repositorio. Sin embargo, algunos atributos operacionales (OTIF, daños, errores documentales) se simulan con distribuciones de probabilidad basadas en benchmarks reales de la industria, ya que estos datos no están disponibles en los archivos CSV base. En un entorno productivo, estos datos provendrían de sistemas TMS, WMS y ERP.

In [15]:
# Ruta a datos
data_dir = root / 'data' / 'raw'

# Cargar datasets necesarios
orders = pd.read_csv(data_dir / 'orders.csv', parse_dates=['date'])
inventory = pd.read_csv(data_dir / 'inventory.csv')
transport = pd.read_csv(data_dir / 'transport_events.csv')
locations = pd.read_csv(data_dir / 'locations.csv')
products = pd.read_csv(data_dir / 'products.csv')

# Enriquecer orders con información de productos
orders = orders.merge(products[['sku', 'unit_cost', 'category']], on='sku', how='left')

print(f"📊 Datasets cargados:")
print(f"   Orders: {len(orders):,} registros")
print(f"   Inventory: {len(inventory):,} registros")
print(f"   Transport: {len(transport):,} registros")
print(f"   Locations: {len(locations):,} ubicaciones")
print(f"   Products: {len(products):,} SKUs")
print(f"   Período: {orders['date'].min()} a {orders['date'].max()}")
print(f"✅ Paso 1 completado")

📊 Datasets cargados:
   Orders: 8,504 registros
   Inventory: 3,000 registros
   Transport: 2,995 registros
   Locations: 30 ubicaciones
   Products: 200 SKUs
   Período: 2024-01-01 00:00:00 a 2024-03-31 00:00:00
✅ Paso 1 completado


---

## 🔢 Paso 2: KPI 1 - OTIF (On-Time In-Full)

**Definición**: % de órdenes entregadas completas y a tiempo.

**Fórmula**: `OTIF = (Órdenes On-Time AND In-Full / Total Órdenes) × 100`

**Meta típica**: > 95% (clase mundial), > 85% (promedio industria)

**Impacto**: Satisfacción del cliente, reducción de reclamos, retención de contratos.

In [16]:
# Simular métricas de entrega basadas en patrones realistas de supply chain
# En producción, estos datos vendrían del TMS (Transportation Management System)

# OTIF varía por canal: B2B más estable, Ecom más variable, Retail intermedio
np.random.seed(42)  # Para reproducibilidad
otif_probabilities = {
    'B2B': 0.94,      # Clientes empresariales: entregas más predecibles
    'Retail': 0.91,   # Tiendas retail: volumen medio, cumplimiento medio
    'Ecom': 0.89      # E-commerce: alta variabilidad, última milla compleja
}

# Asignar OTIF basado en canal con distribución realista
orders['delivered_on_time'] = orders['channel'].apply(
    lambda c: np.random.random() < otif_probabilities.get(c, 0.92)
)

# Entregas incompletas: más común en productos de alta rotación (quiebres de stock)
# Productos Electronics tienen más quiebres (15%) vs. Household (5%)
stockout_prob = orders['category'].map({'Electronics': 0.12, 'Household': 0.05, 'Food': 0.08}).fillna(0.07)
orders['delivered_complete'] = np.random.random(len(orders)) > stockout_prob

# Calcular OTIF
orders['otif'] = orders['delivered_on_time'] & orders['delivered_complete']
otif_rate = orders['otif'].mean() * 100

print(f"📊 OTIF Global: {otif_rate:.1f}%")
print(f"   On-Time Rate: {orders['delivered_on_time'].mean()*100:.1f}%")
print(f"   In-Full Rate: {orders['delivered_complete'].mean()*100:.1f}%")

# OTIF por canal
otif_by_channel = orders.groupby('channel')['otif'].mean() * 100
print(f"\n📦 OTIF por Canal:")
for channel, rate in otif_by_channel.items():
    print(f"   {channel}: {rate:.1f}%")

fig = go.Figure()
fig.add_trace(go.Bar(
    x=otif_by_channel.index,
    y=otif_by_channel.values,
    marker_color=['green' if x >= 95 else 'orange' if x >= 90 else 'red' for x in otif_by_channel.values],
    text=[f"{v:.1f}%" for v in otif_by_channel.values],
    textposition='outside'
))
fig.add_hline(y=95, line_dash="dash", line_color="green", annotation_text="Meta: 95%")
fig.update_layout(
    title="📦 OTIF por Canal de Venta",
    xaxis_title="Canal",
    yaxis_title="OTIF (%)",
    height=400
)
fig.show()

📊 OTIF Global: 84.3%
   On-Time Rate: 91.2%
   In-Full Rate: 92.5%

📦 OTIF por Canal:
   B2B: 87.3%
   Ecom: 82.8%
   Retail: 84.0%


---

## 📈 Paso 3: KPI 2 - Fill Rate (Tasa de Servicio)

**Definición**: % de la demanda servida sin quiebres de stock.

**Fórmula**: `Fill Rate = (Unidades Servidas / Unidades Solicitadas) × 100`

**Meta típica**: > 98%

**Impacto**: Ventas perdidas, erosión de marca, clientes insatisfechos.

In [17]:
# Calcular Fill Rate basado en disponibilidad real de inventario
# En producción: match orders vs. inventory disponible en momento de la orden

# Crear lookup de inventario total por SKU
inventory_available = inventory.groupby('sku')['on_hand'].sum()

# Para cada orden, verificar si había inventario suficiente
# Simplificación: asumimos que el inventario actual representa el promedio del período
orders['inventory_available'] = orders['sku'].map(inventory_available).fillna(0)

# Qty servida: menor entre qty solicitada y disponible (con factor de seguridad 0.8)
# Factor 0.8 porque no todo el inventario está disponible (reservas, tránsito, etc.)
orders['served_qty'] = orders.apply(
    lambda row: min(row['qty'], row['inventory_available'] * 0.8 / (len(orders) / len(inventory))),
    axis=1
)

# Fill Rate global
fill_rate = (orders['served_qty'].sum() / orders['qty'].sum()) * 100

print(f"📊 Fill Rate Global: {fill_rate:.1f}%")

# Fill Rate por categoría (más realista)
fill_by_category = orders.groupby('category').apply(
    lambda g: (g['served_qty'].sum() / g['qty'].sum()) * 100
).sort_values()
print(f"\n📦 Fill Rate por Categoría:")
for cat, rate in fill_by_category.items():
    print(f"   {cat}: {rate:.1f}%")

# Tendencia mensual con variación estacional realista
fill_by_month = orders.set_index('date').resample('M').apply(
    lambda g: (g['served_qty'].sum() / g['qty'].sum()) * 100
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fill_by_month.index,
    y=fill_by_month.values,
    mode='lines+markers',
    name='Fill Rate',
    line=dict(color='blue', width=3),
    fill='tozeroy'
))
fig.add_hline(y=98, line_dash="dash", line_color="green", annotation_text="Meta: 98%")
fig.update_layout(
    title="📊 Fill Rate Mensual",
    xaxis_title="Mes",
    yaxis_title="Fill Rate (%)",
    height=400
)
fig.show()

📊 Fill Rate Global: 100.0%

📦 Fill Rate por Categoría:
   Beverages: 100.0%
   Electronics: 100.0%
   Household: 100.0%
   PersonalCare: 100.0%
   Snacks: 100.0%


---

## 🔄 Paso 4: KPI 3 - Inventory Turns (Rotación de Inventario)

**Definición**: Cuántas veces se vende el inventario promedio en un período.

**Fórmula**: `Turns = (COGS / Inventario Promedio) × (365 / Días)`

**Meta típica**: Depende de industria (retail: 5-8, farma: 12-15, automotriz: 8-10)

**Impacto**: Capital de trabajo, obsolescencia, costos de almacenaje, cash flow.

In [18]:
# Calcular COGS (Cost of Goods Sold) usando unit_cost real de productos
orders['cogs'] = orders['qty'] * orders['unit_cost']
total_cogs = orders['cogs'].sum()

# Inventario promedio valorizado con costos reales
inventory_valued = inventory.merge(products[['sku', 'unit_cost']], on='sku', how='left')
inventory_valued['value'] = inventory_valued['on_hand'] * inventory_valued['unit_cost']
avg_inventory_value = inventory_valued['value'].sum()

# Inventory turns (anualizado)
days_in_period = (orders['date'].max() - orders['date'].min()).days
turns = (total_cogs / avg_inventory_value) * (365 / days_in_period)

print(f"📊 Inventory Turns (anual): {turns:.1f}x")
print(f"   Days of Inventory: {365/turns:.0f} días")
print(f"   COGS (período): ${total_cogs:,.0f}")
print(f"   Inventario promedio: ${avg_inventory_value:,.0f}")
print(f"\n💡 Benchmark industria retail: 5-8 turns/año")

# Crear gauge chart
fig = go.Figure(go.Indicator(
    mode="gauge+number+delta",
    value=turns,
    title={'text': "🔄 Inventory Turns (Anual)"},
    delta={'reference': 6, 'suffix': 'x'},
    gauge={
        'axis': {'range': [None, 12]},
        'bar': {'color': "darkblue"},
        'steps': [
            {'range': [0, 4], 'color': "lightgray"},
            {'range': [4, 8], 'color': "lightblue"},
            {'range': [8, 12], 'color': "lightgreen"}
        ],
        'threshold': {
            'line': {'color': "green", 'width': 4},
            'thickness': 0.75,
            'value': 6
        }
    }
))
fig.update_layout(height=400)
fig.show()

📊 Inventory Turns (anual): 1.9x
   Days of Inventory: 197 días
   COGS (período): $6,437,618
   Inventario promedio: $14,060,666

💡 Benchmark industria retail: 5-8 turns/año


---

## 💰 Paso 5: KPI 4 - Costo Logístico (% de Ventas)

**Definición**: Total de costos logísticos como % de ventas netas.

**Fórmula**: `Logistics Cost % = (Costos Logísticos Totales / Ventas Netas) × 100`

**Meta típica**: 5-8% (varía por industria), <5% clase mundial

**Impacto**: Rentabilidad, competitividad, pricing, margen operativo.

In [19]:
# Calcular costos logísticos basados en reglas de negocio realistas
# Ventas = COGS × Markup (típicamente 1.5-2.5 en retail)
markup_factor = 1.8  # 80% markup sobre costo
total_sales = total_cogs * markup_factor

# Costos logísticos como % de ventas (basado en benchmarks de industria)
# Transporte: varía por distancia, consolidación, modal (2-3% ventas)
# Almacenaje: depende de tipo de producto, rotación (1-2% ventas)
# Manejo: picking, packing, labor (0.8-1.5% ventas)
# Administración: sistemas, overhead (0.5-1% ventas)

logistics_cost = {
    'Transporte': 0.027 * total_sales,      # 2.7% - última milla es costosa
    'Almacenaje': 0.016 * total_sales,      # 1.6% - incluye rent, utilities
    'Manejo': 0.011 * total_sales,          # 1.1% - labor de warehouse
    'Administración': 0.007 * total_sales   # 0.7% - WMS, TMS, overhead
}
total_logistics = sum(logistics_cost.values())
logistics_pct = (total_logistics / total_sales) * 100

print(f"📊 Costo Logístico: {logistics_pct:.1f}% de ventas")
print(f"   Ventas totales: ${total_sales:,.0f}")
print(f"   Costo logístico total: ${total_logistics:,.0f}")
print(f"\n📦 Desglose:")
for component, cost in logistics_cost.items():
    print(f"   {component}: ${cost:,.0f} ({cost/total_sales*100:.2f}%)")

# Desglose en waterfall
fig = go.Figure(go.Waterfall(
    x=list(logistics_cost.keys()) + ['Total'],
    y=[v/total_sales*100 for v in logistics_cost.values()] + [logistics_pct],
    measure=['relative']*len(logistics_cost) + ['total'],
    text=[f"{v:.1f}%" for v in list(logistics_cost.values()) + [logistics_pct]],
    textposition='outside',
    connector={'line': {'color': 'rgb(63, 63, 63)'}}
))
fig.update_layout(
    title="💰 Desglose de Costo Logístico (% ventas)",
    yaxis_title="% de Ventas",
    height=400
)
fig.show()

📊 Costo Logístico: 6.1% de ventas
   Ventas totales: $11,587,712
   Costo logístico total: $706,850

📦 Desglose:
   Transporte: $312,868 (2.70%)
   Almacenaje: $185,403 (1.60%)
   Manejo: $127,465 (1.10%)
   Administración: $81,114 (0.70%)


---

## ⚡ Paso 6: KPI 5 - Perfect Order Rate

**Definición**: % de órdenes perfectas (a tiempo, completa, sin daños, documentación correcta).

**Fórmula**: `Perfect Order = (On-Time AND In-Full AND No-Damage AND Correct-Docs) / Total × 100`

**Meta típica**: > 90% (cada 1% mejora reduce costos ocultos significativamente)

**Impacto**: Costos ocultos (devoluciones, reenvíos, créditos), satisfacción cliente.

In [20]:
# Perfect order = on time + complete + no damage + correct docs
# Tasas de defectos basadas en estadísticas reales de supply chain:

# Daños: más común en productos frágiles (Electronics) vs. duraderos (Household)
damage_prob = orders['category'].map({
    'Electronics': 0.04,   # 4% daños en electrónicos (frágiles)
    'Household': 0.02,     # 2% daños en household (más resistente)
    'Food': 0.03           # 3% daños en food (perecedero)
}).fillna(0.03)
orders['no_damage'] = np.random.random(len(orders)) > damage_prob

# Documentación incorrecta: más común en B2B (facturas, customs) vs. Ecom (simple)
doc_error_prob = orders['channel'].map({
    'B2B': 0.05,      # 5% errores en B2B (documentación compleja)
    'Retail': 0.03,   # 3% errores en retail (media complejidad)
    'Ecom': 0.02      # 2% errores en ecom (automatizado)
}).fillna(0.03)
orders['correct_docs'] = np.random.random(len(orders)) > doc_error_prob

# Perfect Order: todas las condiciones deben cumplirse
orders['perfect'] = orders['otif'] & orders['no_damage'] & orders['correct_docs']

perfect_rate = orders['perfect'].mean() * 100
print(f"📊 Perfect Order Rate: {perfect_rate:.1f}%")
print(f"\n📊 Componentes:")
print(f"   OTIF: {orders['otif'].mean()*100:.1f}%")
print(f"   Sin daño: {orders['no_damage'].mean()*100:.1f}%")
print(f"   Docs correctos: {orders['correct_docs'].mean()*100:.1f}%")

# Pareto de problemas
issues = {
    'Retraso': (~orders['delivered_on_time']).sum(),
    'Incompleto': (~orders['delivered_complete']).sum(),
    'Daño': (~orders['no_damage']).sum(),
    'Documentación': (~orders['correct_docs']).sum()
}
issues_df = pd.DataFrame(list(issues.items()), columns=['Tipo', 'Cantidad'])
issues_df = issues_df.sort_values('Cantidad', ascending=False)
issues_df['Acumulado%'] = (issues_df['Cantidad'].cumsum() / issues_df['Cantidad'].sum() * 100)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Bar(x=issues_df['Tipo'], y=issues_df['Cantidad'], name='Cantidad', marker_color='indianred'),
    secondary_y=False
)
fig.add_trace(
    go.Scatter(x=issues_df['Tipo'], y=issues_df['Acumulado%'], name='% Acumulado', 
               marker_color='blue', mode='lines+markers'),
    secondary_y=True
)
fig.update_layout(title="📊 Pareto de Problemas en Órdenes", height=400)
fig.update_yaxes(title_text="Cantidad de Órdenes", secondary_y=False)
fig.update_yaxes(title_text="% Acumulado", secondary_y=True)
fig.show()

📊 Perfect Order Rate: 79.5%

📊 Componentes:
   OTIF: 84.3%
   Sin daño: 97.3%
   Docs correctos: 96.9%


---

## 📊 Paso 7: Dashboard Consolidado

**Técnica:** Integración multi-KPI con layout de subplots

**Objetivo:** Proveer vista ejecutiva de un vistazo (at-a-glance) con todos los KPIs críticos y sus tendencias.

**Elementos del dashboard:**
- 5 indicadores numéricos con delta vs. meta
- Sparkline de tendencia diaria de órdenes
- Código de colores según performance

In [21]:
# Crear dashboard con subplot
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=('OTIF', 'Fill Rate', 'Inventory Turns', 
                    'Costo Logístico', 'Perfect Order', 'Tendencia Órdenes'),
    specs=[[{'type': 'indicator'}, {'type': 'indicator'}, {'type': 'indicator'}],
           [{'type': 'indicator'}, {'type': 'indicator'}, {'type': 'scatter'}]],
    vertical_spacing=0.15
)

# OTIF
fig.add_trace(go.Indicator(
    mode="number+delta",
    value=otif_rate,
    title={'text': "OTIF (%)"},
    delta={'reference': 95, 'relative': False},
    number={'suffix': '%'}
), row=1, col=1)

# Fill Rate
fig.add_trace(go.Indicator(
    mode="number+delta",
    value=fill_rate,
    title={'text': "Fill Rate (%)"},
    delta={'reference': 98, 'relative': False},
    number={'suffix': '%'}
), row=1, col=2)

# Inventory Turns
fig.add_trace(go.Indicator(
    mode="number+delta",
    value=turns,
    title={'text': "Inventory Turns"},
    delta={'reference': 6, 'relative': False},
    number={'suffix': 'x'}
), row=1, col=3)

# Costo Logístico
fig.add_trace(go.Indicator(
    mode="number+delta",
    value=logistics_pct,
    title={'text': "Costo Log. (% ventas)"},
    delta={'reference': 6, 'relative': False},
    number={'suffix': '%'}
), row=2, col=1)

# Perfect Order
fig.add_trace(go.Indicator(
    mode="number+delta",
    value=perfect_rate,
    title={'text': "Perfect Order (%)"},
    delta={'reference': 90, 'relative': False},
    number={'suffix': '%'}
), row=2, col=2)

# Tendencia órdenes diarias
daily_orders = orders.set_index('date').resample('D').size()
fig.add_trace(go.Scatter(
    x=daily_orders.index,
    y=daily_orders.values,
    mode='lines',
    name='Órdenes/día',
    line=dict(color='blue', width=2),
    fill='tozeroy'
), row=2, col=3)

fig.update_layout(
    title_text="📊 Dashboard Ejecutivo de Supply Chain",
    height=600,
    showlegend=False
)
fig.show()

print("\n✅ Dashboard generado exitosamente")


✅ Dashboard generado exitosamente


---

## 🔧 Paso 8: Funciones Reutilizables

**Técnica:** Encapsulación de lógica de KPIs para reutilización

**Objetivo:** Permitir actualización automática del dashboard con nuevos datos sin reescribir código.

In [22]:
def calculate_otif(orders_df, date_col='date', qty_col='qty'):
    """
    Calculate OTIF (On-Time In-Full) rate.
    
    Args:
        orders_df: DataFrame with orders
        date_col: Column name for date
        qty_col: Column name for quantity
        
    Returns:
        float: OTIF rate as percentage
    """
    if 'otif' not in orders_df.columns:
        raise ValueError("DataFrame must have 'otif' column")
    return orders_df['otif'].mean() * 100


def create_kpi_indicator(value, title, reference, suffix='%'):
    """
    Create a Plotly indicator for KPI display.
    
    Args:
        value: Current KPI value
        title: KPI name
        reference: Target/reference value
        suffix: Unit suffix
        
    Returns:
        go.Indicator: Plotly indicator object
    """
    return go.Indicator(
        mode="number+delta",
        value=value,
        title={'text': title},
        delta={'reference': reference, 'relative': False},
        number={'suffix': suffix}
    )


print("✅ Funciones auxiliares definidas")

✅ Funciones auxiliares definidas


---

## 💡 Paso 9: Insights Adicionales y Análisis de Correlación

**Técnica:** Análisis de correlación entre KPIs y segmentación por canal/categoría

**Objetivo:** Identificar patrones ocultos y oportunidades de mejora específicas

In [23]:
# Análisis por canal: ¿Qué canal tiene mejor desempeño integral?
channel_performance = orders.groupby('channel').agg({
    'otif': 'mean',
    'perfect': 'mean',
    'served_qty': 'sum',
    'qty': 'sum'
}).round(3)
channel_performance['fill_rate'] = (channel_performance['served_qty'] / channel_performance['qty']) * 100
channel_performance = channel_performance[['otif', 'perfect', 'fill_rate']] * 100

print("📊 Performance Integral por Canal:")
print(channel_performance.round(1))

# Análisis por categoría: ¿Qué productos causan más problemas?
category_performance = orders.groupby('category').agg({
    'otif': 'mean',
    'no_damage': 'mean',
    'delivered_complete': 'mean'
}).round(3) * 100

print("\n📦 Performance por Categoría de Producto:")
print(category_performance.round(1))

# Top 5 SKUs con más problemas de OTIF
problematic_skus = orders.groupby('sku').agg({
    'otif': 'mean',
    'qty': 'sum'
}).sort_values('otif').head(5)
problematic_skus['otif'] *= 100

print("\n⚠️ Top 5 SKUs con peor OTIF:")
print(problematic_skus.round(1))

# Impacto financiero de mejorar Perfect Order de actual a 90%
current_imperfect_orders = (~orders['perfect']).sum()
target_improvement = max(0, current_imperfect_orders - len(orders) * 0.1)  # Llevar a 90%

# Costo estimado por orden imperfecta: $50 (reenvíos, créditos, labor)
cost_per_imperfect = 50
potential_savings = target_improvement * cost_per_imperfect

print(f"\n💰 Impacto Financiero de Mejora:")
print(f"   Órdenes imperfectas actuales: {current_imperfect_orders:,.0f}")
print(f"   Reducción potencial a meta 90%: {target_improvement:,.0f} órdenes")
print(f"   Ahorro anual estimado: ${potential_savings:,.0f}")

print("\n✅ Análisis de insights completado")

📊 Performance Integral por Canal:
         otif  perfect  fill_rate
channel                          
B2B     87.30    80.20   10000.00
Ecom    82.80    79.70   10000.00
Retail  84.00    79.00   10000.00

📦 Performance por Categoría de Producto:
              otif  no_damage  delivered_complete
category                                         
Beverages    85.50      97.20               94.30
Electronics  80.20      97.10               88.10
Household    86.70      98.00               94.30
PersonalCare 83.20      97.30               92.00
Snacks       85.00      96.70               92.70

⚠️ Top 5 SKUs con peor OTIF:
           otif  qty
sku                 
SKU-00007 64.30  414
SKU-00114 70.70  308
SKU-00098 72.20  364
SKU-00073 72.70  535
SKU-00099 72.70  449

💰 Impacto Financiero de Mejora:
   Órdenes imperfectas actuales: 1,745
   Reducción potencial a meta 90%: 895 órdenes
   Ahorro anual estimado: $44,730

✅ Análisis de insights completado


---

## 💾 Exportar Resultados

**Formato:** JSON para dashboard metadata, Parquet para análisis histórico y CSV para análisis por canal

In [24]:
# Exportar resultados del dashboard
processed_path = root / 'data' / 'processed' / 'ba05_dashboard'
processed_path.mkdir(parents=True, exist_ok=True)

# Guardar métricas del dashboard
dashboard_metrics = {
    'otif_rate': float(otif_rate),
    'fill_rate': float(fill_rate),
    'inventory_turns': float(turns),
    'logistics_cost_pct': float(logistics_pct),
    'perfect_order_rate': float(perfect_rate),
    'total_cogs': float(total_cogs),
    'total_sales': float(total_sales),
    'avg_inventory_value': float(avg_inventory_value),
    'period_days': int(days_in_period),
    'timestamp': datetime.now().isoformat(),
    'period_start': orders['date'].min().isoformat(),
    'period_end': orders['date'].max().isoformat()
}

import json
with open(processed_path / 'executive_dashboard_metrics.json', 'w') as f:
    json.dump(dashboard_metrics, f, indent=2)

# Exportar datos históricos para análisis
orders.to_parquet(processed_path / 'orders_with_kpis.parquet', index=False)

# Exportar análisis por canal
channel_performance.to_csv(processed_path / 'kpi_by_channel.csv')

# Exportar análisis por categoría
category_performance.to_csv(processed_path / 'kpi_by_category.csv')

print(f"✅ Resultados exportados a {processed_path}")
print(f"   - executive_dashboard_metrics.json")
print(f"   - orders_with_kpis.parquet")

✅ Resultados exportados a f:\GitHub\supply-chain-data-notebooks\data\processed\ba05_dashboard
   - executive_dashboard_metrics.json
   - orders_with_kpis.parquet


---

## ✅ Validaciones

In [25]:
# Validaciones de integridad y lógica de negocio
assert 0 <= otif_rate <= 100, "OTIF debe estar entre 0 y 100%"
assert 0 <= fill_rate <= 100, "Fill Rate debe estar entre 0 y 100%"
assert turns > 0, "Inventory Turns debe ser positivo"
assert 0 <= logistics_pct <= 100, "Costo Logístico % debe estar entre 0 y 100%"
assert 0 <= perfect_rate <= 100, "Perfect Order Rate debe estar entre 0 y 100%"
assert len(orders) > 0, "Dataset de órdenes no puede estar vacío"

# Validaciones de rangos realistas (warnings si están fuera de benchmarks)
print("🔍 Validaciones de rangos vs. benchmarks de industria:")

if otif_rate < 85:
    print(f"   ⚠️ OTIF {otif_rate:.1f}% está por debajo del promedio industria (85%)")
elif otif_rate >= 95:
    print(f"   ✅ OTIF {otif_rate:.1f}% es clase mundial (>95%)")
else:
    print(f"   ✓ OTIF {otif_rate:.1f}% está en rango aceptable (85-95%)")

if fill_rate < 95:
    print(f"   ⚠️ Fill Rate {fill_rate:.1f}% requiere atención (<95%)")
elif fill_rate >= 98:
    print(f"   ✅ Fill Rate {fill_rate:.1f}% es excelente (>98%)")
else:
    print(f"   ✓ Fill Rate {fill_rate:.1f}% está en rango aceptable (95-98%)")

if turns < 4:
    print(f"   ⚠️ Inventory Turns {turns:.1f}x es bajo - revisar slow movers (<4x)")
elif turns > 10:
    print(f"   ⚠️ Inventory Turns {turns:.1f}x es muy alto - riesgo de stockouts (>10x)")
else:
    print(f"   ✅ Inventory Turns {turns:.1f}x está en rango óptimo retail (4-10x)")

if logistics_pct > 8:
    print(f"   ⚠️ Costo Logístico {logistics_pct:.1f}% es alto (>8%)")
elif logistics_pct < 5:
    print(f"   ✅ Costo Logístico {logistics_pct:.1f}% es competitivo (<5%)")
else:
    print(f"   ✓ Costo Logístico {logistics_pct:.1f}% está en rango aceptable (5-8%)")

if perfect_rate < 85:
    print(f"   ⚠️ Perfect Order {perfect_rate:.1f}% requiere mejora urgente (<85%)")
elif perfect_rate >= 90:
    print(f"   ✅ Perfect Order {perfect_rate:.1f}% es clase mundial (>90%)")
else:
    print(f"   ✓ Perfect Order {perfect_rate:.1f}% está en rango aceptable (85-90%)")

print("\n✅ Validaciones pasadas")
print("✅ Notebook BA-05 completado: Dashboard ejecutivo con 5 KPIs críticos generado exitosamente")

🔍 Validaciones de rangos vs. benchmarks de industria:
   ⚠️ OTIF 84.3% está por debajo del promedio industria (85%)
   ✅ Fill Rate 100.0% es excelente (>98%)
   ⚠️ Inventory Turns 1.9x es bajo - revisar slow movers (<4x)
   ✓ Costo Logístico 6.1% está en rango aceptable (5-8%)
   ⚠️ Perfect Order 79.5% requiere mejora urgente (<85%)

✅ Validaciones pasadas
✅ Notebook BA-05 completado: Dashboard ejecutivo con 5 KPIs críticos generado exitosamente


---

## 📚 Resumen Técnico y Referencias



### 🎯 Resultados Clave

Este análisis implementa un dashboard ejecutivo de supply chain consolidando 5 KPIs críticos con visualizaciones interactivas tipo semáforo.

**KPIs calculados:**
1. **OTIF (On-Time In-Full)**: `% órdenes entregadas completas y a tiempo = (órdenes_on_time AND complete) / total_órdenes × 100`
2. **Fill Rate**: `% demanda servida = qty_servida / qty_solicitada × 100`
3. **Inventory Turns**: `Rotación anual = (COGS / inventario_promedio) × (365 / días_período)`
4. **Logistics Cost %**: `Costo logístico como % ventas = costo_log_total / ventas_netas × 100`
5. **Perfect Order Rate**: `% órdenes perfectas = (on_time AND complete AND no_damage AND correct_docs) / total × 100`

**Hallazgos típicos:**
- OTIF enterprise benchmark: 95%+ (clase mundial), 85-95% (promedio industria)
- Fill Rate meta: 98%+ (productos A), 95%+ (productos B/C)
- Inventory Turns varía por industria: Retail 5-8x, Farma 12-15x, Automotriz 8-10x
- Logistics Cost %: 5-8% típico, <5% clase mundial
- Perfect Order: >90% meta, cada 1% mejora reduce costos ocultos significativamente

**Segmentación/Clasificación:**
- **Verde (meta alcanzada)**: Mantener operación, monitorear tendencias
- **Amarillo (cerca de meta)**: Acciones preventivas, análisis drill-down
- **Rojo (fuera de meta)**: Acción correctiva inmediata, root cause analysis

### 🔬 Metodología

**Modelo principal:**

$$
\text{OTIF} = \frac{\text{Órdenes On-Time AND In-Full}}{\text{Total Órdenes}} \times 100
$$

$$
\text{Inventory Turns} = \frac{\text{COGS}}{\text{Inventario Promedio}} \times \frac{365}{\text{Días Período}}
$$

$$
\text{Perfect Order Rate} = \frac{\text{Órdenes sin defectos}}{\text{Total Órdenes}} \times 100
$$

**Técnica aplicada:**
- Business Intelligence con visualizaciones ejecutivas interactivas (Plotly)
- Diseño balanceado de métricas: operacional (OTIF, Fill Rate) + financiero (Logistics Cost) + cliente (Perfect Order)
- Sistema de alertas visuales con código de colores (verde/amarillo/rojo)
- Análisis Pareto para priorización de problemas

### 📖 Aplicaciones Prácticas

1. **S&OP meetings semanales:**
   - Revisión rápida de performance en 15 minutos vs. 3 horas anterior
   - Dashboard proyectado durante reunión, drill-down interactivo en anomalías
   - Decisiones data-driven con métricas actualizadas

2. **Board reporting mensual:**
   - Narrativa ejecutiva con 5 KPIs core sin "bajar a los detalles"
   - Benchmarking vs. metas y período anterior
   - Identificación de iniciativas que requieren atención

3. **Crisis management:**
   - Refresh en vivo durante disrupciones (huelgas, desastres naturales)
   - Monitoreo real-time de impacto en KPIs
   - Validación de efectividad de acciones correctivas

### 🔗 Referencias

1. **APICS (2021)**. *Supply Chain Operations Reference (SCOR) Model*. ASCM.
   - Framework estándar para KPIs de supply chain, definiciones de OTIF y Perfect Order

2. **Chopra, S. & Meindl, P. (2019)**. *Supply Chain Management: Strategy, Planning, and Operation* (7th ed.). Pearson.
   - Fundamentos de métricas operacionales y financieras de supply chain

3. **Aberdeen Group (2020)**. *Executive Dashboard Best Practices for Supply Chain Leaders*. Aberdeen Research.
   - Diseño de dashboards ejecutivos, benchmarks de industria

### 💡 Extensiones Futuras

- Integrar datos en tiempo real vía APIs (ERP, WMS, TMS)
- Implementar alertas automáticas (email/Slack) cuando KPI sale de meta
- Drill-down dinámico por región/producto/cliente/canal
- Benchmarking contra competencia con datos externos
- Machine Learning para forecasting de KPIs y detección de anomalías
- Integración con herramientas BI enterprise (Power BI, Tableau)

---

**Autor**: lraigosov (@LuisRai)  
**Fecha**: 2024 a la actualidad  
**Versión**: 1.0  
**Tags**: `#dashboard` `#kpis` `#executive-reporting` `#supply-chain` `#bi`

---

<div style="width: 100%; clear: both; margin: 0 0 20px 0; border-top: 1px solid #eaecef; padding-top: 24px;"><div style="display: flex; justify-content: space-between; align-items: center; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif;"><div style="flex: 1; text-align: left;"><a href="BA-04-supplier_performance.ipynb" style="text-decoration: none; color: #0366d6; font-size: 14px; font-weight: 600; transition: color 0.2s;">← Anterior: BA-04-supplier_performance.ipynb</a></div><div style="flex: 1; text-align: center; font-size: 14px;"><a href="../../README.md" style="color: #0366d6; text-decoration: none; font-weight: 600; margin: 0 10px;">📑 Índice</a><span style="color: #6a737d;">|</span><a href="../../config/notebooks_index.yml" style="color: #0366d6; text-decoration: none; font-weight: 600; margin: 0 10px;">📋 Catálogo</a></div><div style="flex: 1; text-align: right;"><span style="color: #6a737d; font-size: 14px; cursor: default;">Siguiente →</span></div></div></div>

